## ML_1031: Gradient Norm Clipping

**Difficulty**: Easy | **TorchCode**: #21

### Problem
Implement gradient norm clipping. Compute the global L2 norm of all gradients, then scale all gradients down if the total norm exceeds `max_norm`. Return the pre-clip total norm.

### Formula
$$\|g\|_2 = \sqrt{\sum_i \|\nabla_{\theta_i}\|^2}, \quad g_i \leftarrow g_i \cdot \frac{\text{max\_norm}}{\|g\|_2} \text{ if } \|g\|_2 > \text{max\_norm}$$

In [ ]:
import torch

def clip_grad_norm(parameters, max_norm):
    parameters = [p for p in parameters if p.grad is not None]
    total_norm = torch.sqrt(sum(p.grad.norm() ** 2 for p in parameters))
    clip_coef = max_norm / (total_norm + 1e-6)
    if clip_coef < 1:
        for p in parameters:
            p.grad.mul_(clip_coef)
    return total_norm.item()

In [ ]:
import torch

# --- Test 1: Clips to max_norm ---
p1 = torch.randn(10, requires_grad=True)
p2 = torch.randn(10, requires_grad=True)
(p1 * 10).sum().backward()
(p2 * 10).sum().backward()
clip_grad_norm([p1, p2], max_norm=1.0)
new_norm = torch.sqrt(p1.grad.norm()**2 + p2.grad.norm()**2).item()
assert new_norm <= 1.0 + 1e-5

# --- Test 2: Returns original norm ---
p = torch.randn(10, requires_grad=True)
(p * 3).sum().backward()
expected = p.grad.norm().item()
returned = clip_grad_norm([p], max_norm=100.0)
assert abs(returned - expected) < 1e-4

# --- Test 3: No change when norm < max_norm ---
p = torch.randn(4, requires_grad=True)
(p * 0.001).sum().backward()
grad_before = p.grad.clone()
clip_grad_norm([p], max_norm=100.0)
assert torch.equal(p.grad, grad_before)

# --- Test 4: Preserves direction ---
torch.manual_seed(0)
p = torch.randn(100, requires_grad=True)
(p * 10).sum().backward()
dir_before = p.grad / p.grad.norm()
clip_grad_norm([p], max_norm=1.0)
dir_after = p.grad / p.grad.norm()
assert torch.allclose(dir_before, dir_after, atol=1e-5)

print('All tests passed!')